### Filter & Attribute Vulnerable Hosts
Reads raw scan data (JSON), keeps only hosts that have at least one real
CVE (a genuine, known vulnerability — not just an open port), attributes
each host to its owning company, and writes two CSVs: `vulnerable_hosts.csv`
and `vulnerable_cves.csv`.

> Public data only — no scanning/exploitation.

In [ ]:
# imports & configuration
import json, os, csv, glob

try:
    import orjson
    def _loads(b): return orjson.loads(b)
except Exception:
    def _loads(b): return json.loads(b)

# PROJECT_ROOT = original repo root, two levels above this notebook (pyspark_files/../..).
# Used only to read the raw input JSON (not duplicated into this new repo).
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

# INPUT_PATH: NDJSON, JSON array, or a glob (e.g. "data/raw_data/*.json")
INPUT_PATH = os.path.join(PROJECT_ROOT, "data", "raw_data", "test_scans.json")

# OUTPUT_ROOT = this repo's root, one level above this notebook (pyspark_files/..).
OUTPUT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# OUTPUT_DIR: where vulnerable_hosts.csv and vulnerable_cves.csv are written
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, "data", "filtered_data")

# True = keep only confirmed CVEs (drops version-inferred ones)
VERIFIED_ONLY = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"INPUT_PATH   = {INPUT_PATH}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"input exists = {bool(glob.glob(INPUT_PATH))}")


### 1. Input Reader
Reads NDJSON or a JSON array, one record (dict) at a time.

In [ ]:
# robust JSON reader: handles NDJSON or a JSON array
def iter_json_records(path_or_glob):
    """Yield one dict per record from every file matching the glob.
    Detects format by peeking the first non-whitespace char: '[' => array."""
    for path in sorted(glob.glob(path_or_glob)):
        first = b""
        with open(path, "rb") as fh:
            while True:
                ch = fh.read(1)
                if not ch:
                    break
                if not ch.isspace():
                    first = ch
                    break
        if first == b"[":
            with open(path, "rb") as fh:
                try:
                    data = _loads(fh.read())
                except Exception as e:
                    print(f"[warn] could not parse array {path}: {e}")
                    continue
            if isinstance(data, list):
                for rec in data:
                    if isinstance(rec, dict):
                        yield rec
        else:
            with open(path, "r", encoding="utf-8", errors="replace") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = _loads(line)
                    except Exception:
                        continue
                    if isinstance(rec, dict):
                        yield rec


### 2. Vulnerability Classifier
What we're filtering for: a record is kept only if it has at least one real
CVE — either listed in its `vulns` field (CVE id -> `{cvss, epss, verified,
...}`) or flagged by a cve/vuln tag. Everything else (an open port with no
known CVE) is dropped.

`verified=true` means the CVE was confirmed directly; `false` means it was
only inferred from the version banner.

In [ ]:
# vulnerability classifier
def _tags(rec):
    t = rec.get("tags")
    if isinstance(t, list):
        return [str(x).lower() for x in t]
    if isinstance(t, str):
        return [t.lower()]
    return []

def _tag_vuln_signal(rec):
    return any(("cve" in t or "vuln" in t) for t in _tags(rec))

def _parse_vulns(vulns):
    cves, verified, max_cvss = [], [], None
    if isinstance(vulns, dict):
        for cve, meta in vulns.items():
            cves.append(cve)
            if isinstance(meta, dict):
                if meta.get("verified") is True:
                    verified.append(cve)
                c = meta.get("cvss")
                if isinstance(c, (int, float)):
                    max_cvss = c if max_cvss is None else max(max_cvss, c)
    return sorted(cves), verified, max_cvss

def severity_band(cvss):
    if cvss is None:
        return "Unknown"
    if cvss >= 9.0:
        return "Critical"
    if cvss >= 7.0:
        return "High"
    if cvss >= 4.0:
        return "Medium"
    return "Low"

def classify_vuln(rec):
    """Return summary dict if genuinely vulnerable else None; attach _vuln_* fields."""
    vulns = rec.get("vulns")
    has_vulns_dict = isinstance(vulns, dict) and len(vulns) > 0
    if not has_vulns_dict and not _tag_vuln_signal(rec):
        return None
    cves, verified_cves, max_cvss = _parse_vulns(vulns) if has_vulns_dict else ([], [], None)
    if VERIFIED_ONLY and not verified_cves:
        return None
    bucket = "verified" if verified_cves else "unverified"
    rec["_vuln_cves"]          = cves
    rec["_vuln_verified_cves"] = verified_cves
    rec["_vuln_max_cvss"]      = max_cvss
    rec["_vuln_count"]         = len(cves)
    rec["_vuln_bucket"]        = bucket
    rec["_vuln_severity"]      = severity_band(max_cvss)
    return {"bucket": bucket, "cves": cves}


### 3. Company Attribution
`org`/`isp` usually name the **hosting provider** (AWS, Cloudflare), not the
real owner. We rank signals by confidence: TLS cert Organization > cert
CN/SAN domain > reverse-DNS hostname > domains > HTTP host > non-provider
org > isp/asn.

In [ ]:
# company attribution
import re

HOSTING_PROVIDERS = {
    "amazon","aws","google","cloudflare","microsoft","azure","akamai",
    "digitalocean","ovh","hetzner","linode","godaddy","comcast","verizon",
    "telecom","hosting","datacenter","data center","colo","leaseweb","vultr",
    "oracle cloud","alibaba","tencent","fastly","rackspace","scaleway",
    "contabo","namecheap","gandi","level3","cogent","hurricane electric",
    "telia","at&t","charter","orange","vodafone","china unicom","china mobile",
}
MULTI_SUFFIXES = {
    "co.uk","org.uk","gov.uk","ac.uk","co.jp","co.in","com.au","net.au",
    "org.au","com.br","com.cn","com.mx","co.za","co.nz","com.sg","com.hk",
    "co.kr","com.tr","com.tw","co.il","com.ar",
}

def registrable_domain(host):
    if not host or not isinstance(host, str):
        return None
    host = host.strip().lower().rstrip(".")
    host = re.sub(r"^[a-z]+://", "", host).split("/")[0].split(":")[0]
    if not host or re.match(r"^\d+\.\d+\.\d+\.\d+$", host):
        return None
    labels = host.split(".")
    if len(labels) < 2:
        return None
    last2 = ".".join(labels[-2:])
    last3 = ".".join(labels[-3:]) if len(labels) >= 3 else None
    if last2 in MULTI_SUFFIXES and last3:
        return last3
    return last2

def company_name_from_domain(domain):
    if not domain:
        return None
    brand = re.sub(r"[-_]+", " ", domain.split(".")[0])
    return " ".join(w.capitalize() for w in brand.split() if w)

def is_provider(org):
    if not org:
        return False
    low = org.lower()
    return any(p in low for p in HOSTING_PROVIDERS)

def _get(d, *path):
    cur = d
    for k in path:
        if isinstance(cur, dict):
            cur = cur.get(k)
        else:
            return None
    return cur

def _san_domains(ssl):
    out = []
    cert = _get(ssl, "cert") or {}
    san = cert.get("subject_alt_name")
    if isinstance(san, str):
        out += re.findall(r"DNS:([^,\s]+)", san) or [san]
    elif isinstance(san, list):
        out += [str(x) for x in san]
    exts = cert.get("extensions")
    if isinstance(exts, list):
        for e in exts:
            if isinstance(e, dict) and "altname" in str(e.get("name","")).lower():
                out += re.findall(r"DNS:([^,\s]+)", str(e.get("data","")))
    return out

def extract_company_signals(rec):
    signals = []
    ssl = rec.get("ssl") or {}
    org_o = _get(ssl, "cert", "subject", "O")
    if org_o and not is_provider(org_o):
        signals.append({"source":"ssl_cert_O","value":org_o,"domain":None,"confidence":0.95})
    cn = _get(ssl, "cert", "subject", "CN")
    cn_hosts = ([cn] if cn else []) + _san_domains(ssl)
    for h in cn_hosts:
        dom = registrable_domain(str(h).lstrip("*."))
        if dom:
            signals.append({"source":"ssl_cert_cn_san","value":h,"domain":dom,"confidence":0.90})
    for h in (rec.get("hostnames") or []):
        dom = registrable_domain(h)
        if dom:
            signals.append({"source":"hostname","value":h,"domain":dom,"confidence":0.80})
    for d in (rec.get("domains") or []):
        dom = registrable_domain(d) or (d if "." in str(d) else None)
        if dom:
            signals.append({"source":"domains","value":d,"domain":dom,"confidence":0.80})
    http = rec.get("http") or {}
    host_hdr = http.get("host")
    if host_hdr:
        dom = registrable_domain(host_hdr)
        if dom:
            signals.append({"source":"http_host","value":host_hdr,"domain":dom,"confidence":0.70})
    org = rec.get("org")
    if org:
        if is_provider(org):
            signals.append({"source":"org_provider","value":org,"domain":None,"confidence":0.20})
        else:
            signals.append({"source":"org","value":org,"domain":None,"confidence":0.50})
    isp = rec.get("isp")
    if isp:
        signals.append({"source":"isp","value":isp,"domain":None,"confidence":0.15})
    asn = rec.get("asn")
    if asn:
        signals.append({"source":"asn","value":str(asn),"domain":None,"confidence":0.10})
    signals.sort(key=lambda s: s["confidence"], reverse=True)
    return signals

def attribute_company(rec):
    signals = extract_company_signals(rec)
    company = company_domain = source = None
    confidence = 0.0
    provider_flag = False
    for s in signals:
        if s["source"] == "ssl_cert_O":
            company, source, confidence = s["value"], s["source"], s["confidence"]
            break
        if s["domain"]:
            company = company_name_from_domain(s["domain"])
            company_domain, source, confidence = s["domain"], s["source"], s["confidence"]
            break
        if s["source"] == "org":
            company, source, confidence = s["value"], s["source"], s["confidence"]
            break
    if company is None:
        for s in signals:
            if s["source"] in ("org_provider","isp","asn"):
                company, source, confidence = s["value"], s["source"], s["confidence"]
                provider_flag = True
                break
    if source in ("org_provider","isp","asn"):
        provider_flag = True
    rec["_company"]            = company
    rec["_company_domain"]     = company_domain
    rec["_company_source"]     = source
    rec["_company_confidence"] = confidence
    rec["_company_is_provider"]= provider_flag
    return rec


### 4. Run — Write `vulnerable_hosts.csv` + `vulnerable_cves.csv`

In [ ]:
# streaming runner: writes both CSVs in a single pass over the input
HOST_COLS = ["ip_str","port","transport","product","version","os","org","isp",
             "asn","timestamp","country","city","_company","_company_domain",
             "_company_source","_company_confidence","_company_is_provider",
             "_vuln_bucket","_vuln_count","_vuln_max_cvss","_vuln_severity",
             "_vuln_cves","_vuln_verified_cves"]
CVE_COLS = ["ip_str","port","_company","_company_domain","cve","verified",
            "cvss","epss","severity","summary"]

def run(input_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    hosts_path = os.path.join(output_dir, "vulnerable_hosts.csv")
    cves_path  = os.path.join(output_dir, "vulnerable_cves.csv")
    total = kept = verified = unverified = 0

    with open(hosts_path, "w", newline="", encoding="utf-8") as hf, \
         open(cves_path, "w", newline="", encoding="utf-8") as cf:
        hw = csv.DictWriter(hf, fieldnames=HOST_COLS, extrasaction="ignore")
        cw = csv.DictWriter(cf, fieldnames=CVE_COLS, extrasaction="ignore")
        hw.writeheader(); cw.writeheader()

        for rec in iter_json_records(input_path):
            total += 1
            summary = classify_vuln(rec)
            if summary is None:
                continue
            attribute_company(rec)               # adds company name
            kept += 1
            if summary["bucket"] == "verified":
                verified += 1
            else:
                unverified += 1

            # host row
            row = {c: rec.get(c) for c in HOST_COLS}
            row["country"] = _get(rec, "location", "country_name")
            row["city"]    = _get(rec, "location", "city")
            row["_vuln_cves"] = ";".join(rec.get("_vuln_cves", []))
            row["_vuln_verified_cves"] = ";".join(rec.get("_vuln_verified_cves", []))
            hw.writerow(row)

            # one row per CVE
            vulns = rec.get("vulns") or {}
            if isinstance(vulns, dict):
                for cve, meta in vulns.items():
                    meta = meta if isinstance(meta, dict) else {}
                    cw.writerow({
                        "ip_str": rec.get("ip_str"), "port": rec.get("port"),
                        "_company": rec.get("_company"),
                        "_company_domain": rec.get("_company_domain"),
                        "cve": cve, "verified": meta.get("verified"),
                        "cvss": meta.get("cvss"), "epss": meta.get("epss"),
                        "severity": severity_band(meta.get("cvss")),
                        "summary": (meta.get("summary") or "")[:200],
                    })
            if total % 100000 == 0:
                print(f"  ...scanned {total:,} | kept {kept:,}")

    print(f"\nDone. Scanned {total:,}, kept {kept:,} (verified={verified:,}, unverified={unverified:,}).")
    print(f"CSVs -> {output_dir}/vulnerable_hosts.csv, vulnerable_cves.csv")
    return kept


In [ ]:
# execute (safe if input not present yet)
if glob.glob(INPUT_PATH):
    run(INPUT_PATH, OUTPUT_DIR)
else:
    print(f"No input found at '{INPUT_PATH}'. Set INPUT_PATH (top cell) and re-run.")


### Usage
1. `pip install orjson` (optional, faster JSON parsing)
2. Set `INPUT_PATH` to your JSON file or glob
3. Run All

Produces `vulnerable_hosts.csv` (1 row/host) and `vulnerable_cves.csv`
(1 row per host+CVE) in `OUTPUT_DIR`. Lead scoring (`03_score_leads.ipynb`)
and distinct companies (`04_distinct_companies.ipynb`) consume these next.